# Exploring San Francisco Modifier Data
**Author:** Ayemhenre Isikhuemhen  
**Datasets:** ACS Commute Departure Times · LODES Origin-Destination · Decennial Census Population · SFMTA Ridership · SF Land Use 2023  
**Source:** U.S. Census Bureau (ACS / Decennial), U.S. Census Bureau LODES, San Francisco Municipal Transportation Agency (SFMTA), SF Planning Department  

**File Description:** This notebook explores a collection of San Francisco modifier datasets used to enrich and contextualize urban analysis. The files span demographic data (population composition and commute timing), employment origin-destination flows, transit ridership trends, and parcel-level land use. The objective is to examine each file's structure, understand its contents, and build supporting visualizations to inform downstream modelling and planning work.

## Setup
Setup for the modifier data notebook: Import Libraries & File Paths

In [ ]:
# Library Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print("Libraries Imported")

In [ ]:
# File Paths

# Update BASE_DIR if running on a different machine.
BASE_DIR = r"C:\Users\Owner\Documents\GitHub\Valmistada\Data Exploration\Modifier Data\San Francisco"

import os
paths = {
    "commute_time"   : os.path.join(BASE_DIR, "ACSDT1Y2024.C08132-2026-03-26T222031.csv"),
    "origin_dest"    : os.path.join(BASE_DIR, "ca_od_main_JT05_2023.csv"),
    "population"     : os.path.join(BASE_DIR, "DECENNIALPL2020.P1-2026-03-26T215153.csv"),
    "ridership"      : os.path.join(BASE_DIR, "RidershipbyRouteTableDownload.csv"),
    "land_use"       : os.path.join(BASE_DIR, "San_Francisco_Land_Use_-_2023_20260327.csv"),
}

# Load all files
dfs = {name: pd.read_csv(path) for name, path in paths.items()}

print("Files Loaded:")
for name, df in dfs.items():
    print(f"  {name:<20} :  {df.shape[0]:>7,} rows  ×  {df.shape[1]:>3} cols")

## File 1: ACS Commute Departure Times
**File:** `ACSDT1Y2024.C08132-2026-03-26T222031.csv`  
**Description:** American Community Survey (ACS) 1-Year 2024 estimates for San Francisco County. Table C08132 captures the time of departure to work, broken into 30-minute intervals from midnight through the morning peak and beyond. Each row provides an estimate and a margin of error for the number of workers departing within that window. This file contextualises the temporal demand on the transit network and informs peak-period service planning.

In [ ]:
# DF Setup
commute_df = dfs["commute_time"]

In [ ]:
# Shape and Columns
print(f"Shape        : {commute_df.shape}")
print(f"Columns      : {list(commute_df.columns)}")

In [ ]:
# Display the Commute Time DataFrame
display(commute_df)

In [ ]:
# Clean and Parse

ct = commute_df.copy()
ct.columns = ["label", "estimate", "margin_of_error"]

# Drop the total row and strip whitespace from labels
ct = ct[~ct["label"].str.contains("Total", na=False)].copy()
ct["label"] = ct["label"].str.strip()

# Convert estimate to numeric
ct["estimate"] = pd.to_numeric(ct["estimate"].str.replace(",", "").str.strip(), errors="coerce")
ct["moe"] = pd.to_numeric(
    ct["margin_of_error"].str.replace(",", "").str.replace("±", "").str.strip(), errors="coerce"
)
ct = ct.dropna(subset=["estimate"])

print(f"Departure Time Bands : {len(ct)}")
print(f"Total Workers        : {ct['estimate'].sum():,}")

In [ ]:
# Visualization: Departure Time Distribution

fig, ax = plt.subplots(figsize=(12, 5))
colors_bar = sns.color_palette("Blues_r", len(ct))
bars = ax.bar(range(len(ct)), ct["estimate"], color=colors_bar, edgecolor="white")
ax.errorbar(range(len(ct)), ct["estimate"], yerr=ct["moe"],
            fmt='none', color='#333', capsize=3, linewidth=0.8, alpha=0.6)
ax.set_xticks(range(len(ct)))
ax.set_xticklabels(ct["label"], rotation=45, ha="right", fontsize=7)
ax.set_title("SF County — Workers by Departure Time to Work (ACS 2024)", fontsize=13, fontweight="bold")
ax.set_ylabel("Estimated Workers")
ax.grid(axis="y", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

## File 2: LODES Origin-Destination (JT05 2023)
**File:** `ca_od_main_JT05_2023.csv`  
**Description:** Longitudinal Employer-Household Dynamics (LEHD) Origin-Destination Employment Statistics (LODES) for California, Job Type 05 (Federal Primary Jobs), reference year 2023. Each row represents a worker flow between a home Census block (`h_geocode`) and a work Census block (`w_geocode`). Columns encode total jobs (S000) and breakdowns by age (SA01–SA03), earnings tier (SE01–SE03), and industry sector (SI01–SI03). This file is central to understanding commute origins and the spatial distribution of the San Francisco workforce.

In [ ]:
# DF Setup
od_df = dfs["origin_dest"]

In [ ]:
# Shape and Columns of Origin-Destination Data
print(f"Total Records  : {len(od_df):,}")
print(f"Columns        : {list(od_df.columns)}")

In [ ]:
# Display Origin-Destination (head)
display(od_df.head(5))

In [ ]:
# Notable Statistics
print(f"Unique Work Blocks   : {od_df['w_geocode'].nunique():,}")
print(f"Unique Home Blocks   : {od_df['h_geocode'].nunique():,}")
print(f"Total Jobs (S000)    : {od_df['S000'].sum():,}")
print()
print("Age Breakdown:")
print(f"  Age 29 or younger (SA01) : {od_df['SA01'].sum():,}")
print(f"  Age 30 to 54      (SA02) : {od_df['SA02'].sum():,}")
print(f"  Age 55 or older   (SA03) : {od_df['SA03'].sum():,}")
print()
print("Earnings Breakdown:")
print(f"  Up to $1,250/mo   (SE01) : {od_df['SE01'].sum():,}")
print(f"  $1,251–$3,333/mo  (SE02) : {od_df['SE02'].sum():,}")
print(f"  More than $3,333  (SE03) : {od_df['SE03'].sum():,}")

In [ ]:
# Visualization: Jobs by Earnings Tier

earnings_labels = ["≤ $1,250/mo (SE01)", "$1,251–$3,333/mo (SE02)", "> $3,333/mo (SE03)"]
earnings_vals   = [od_df['SE01'].sum(), od_df['SE02'].sum(), od_df['SE03'].sum()]

fig, ax = plt.subplots(figsize=(7, 4))
colors = ["#42A5F5", "#1565C0", "#0D47A1"]
bars = ax.bar(earnings_labels, earnings_vals, color=colors, edgecolor="white", width=0.5)
for bar, val in zip(bars, earnings_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 20,
            f"{val:,}", ha="center", fontsize=10, fontweight="bold")
ax.set_title("LODES — Jobs by Earnings Tier (JT05 2023)", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Jobs")
ax.grid(axis="y", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Visualization: Jobs by Age Group

age_labels = ["≤ 29 (SA01)", "30–54 (SA02)", "≥ 55 (SA03)"]
age_vals   = [od_df['SA01'].sum(), od_df['SA02'].sum(), od_df['SA03'].sum()]

colors_age = ["#FF9800", "#F57C00", "#E65100"]
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(age_labels, age_vals, color=colors_age, edgecolor="white", width=0.5)
for bar, val in zip(bars, age_vals):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 20,
            f"{val:,}", ha="center", fontsize=10, fontweight="bold")
ax.set_title("LODES — Jobs by Age Group (JT05 2023)", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Jobs")
ax.grid(axis="y", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

## File 3: Decennial Census — Population by Race (2020)
**File:** `DECENNIALPL2020.P1-2026-03-26T215153.csv`  
**Description:** Table P1 from the 2020 Decennial Census (Redistricting Data / PL 94-171). Reports the total population of San Francisco city with a breakdown by single-race and multi-race categories. This is the official population count used for apportionment and redistricting. It provides a demographic baseline for understanding who lives in the city and informs equity-focused transit and land-use analysis.

In [ ]:
# DF Setup
pop_df = dfs["population"]

In [ ]:
# Shape and Columns
print(f"Shape        : {pop_df.shape}")
print(f"Columns      : {list(pop_df.columns)}")

In [ ]:
# Display Population DataFrame
display(pop_df)

In [ ]:
# Clean and Parse

pop = pop_df.copy()
pop.columns = ["label", "value"]
pop["label"] = pop["label"].str.strip()
pop["value"] = pd.to_numeric(pop["value"].str.replace(",", "").str.strip(), errors="coerce")

total_pop = pop[pop["label"] == "Total:"]["value"].iloc[0]

# Single-race subcategories (indented with 8 spaces = two levels in)
race_rows = pop[pop["label"].str.startswith("        ") & ~pop["label"].str.startswith("          ")].copy()
race_rows["label"] = race_rows["label"].str.strip()

print(f"Total Population     : {int(total_pop):,}")
print(f"Race Categories      : {len(race_rows)}")

In [ ]:
# Visualization: Population by Race — Horizontal Bar

race_sorted = race_rows.sort_values("value", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
colors_pop = sns.color_palette("Blues", len(race_sorted))
ax.barh(race_sorted["label"], race_sorted["value"],
        color=colors_pop, edgecolor="white")
for i, (val, label) in enumerate(zip(race_sorted["value"], race_sorted["label"])):
    ax.text(val + 500, i, f"{int(val):,}", va="center", fontsize=9)
ax.set_title("San Francisco — Population by Race (2020 Decennial Census)",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Population")
ax.grid(axis="x", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

## File 4: SFMTA Ridership by Route
**File:** `RidershipbyRouteTableDownload.csv`  
**Description:** Monthly average daily boardings by route, service category, and day-of-week published by the San Francisco Municipal Transportation Agency. Each row represents one route–month–service-day combination. The data spans multiple years, allowing trend analysis across pre-pandemic, pandemic, and recovery periods. Service categories include Frequent Local, Rapid, Express, and others. This file links directly to the GTFS route network and provides empirical demand data for ridership modelling.

In [ ]:
# DF Setup
ridership_df = dfs["ridership"]

In [ ]:
# Shape and Columns
print(f"Total Records  : {len(ridership_df):,}")
print(f"Columns        : {list(ridership_df.columns)}")

In [ ]:
# Clean and Parse

rd = ridership_df.copy()
rd.columns = [c.strip() for c in rd.columns]
rd["Average Daily Boardings"] = pd.to_numeric(
    rd["Average Daily Boardings"].astype(str).str.replace(",", "").str.strip(), errors="coerce"
)
rd["Month"] = pd.to_datetime(rd["Month"], format="%B %Y", errors="coerce")
rd = rd.dropna(subset=["Average Daily Boardings", "Month"])

print(f"Unique Routes          : {rd['Route'].nunique()}")
print(f"Date Range             : {rd['Month'].min().strftime('%b %Y')} → {rd['Month'].max().strftime('%b %Y')}")
print(f"Service Categories     : {rd['Service Category'].nunique()}")
print(f"Days of Week Tracked   : {sorted(rd['Service Day of the Week'].unique())}")

In [ ]:
# Display Ridership (head)
display(rd.head(10))

In [ ]:
# Visualization: System-Wide Monthly Ridership Trend

monthly = rd.groupby("Month")["Average Daily Boardings"].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly["Month"], monthly["Average Daily Boardings"],
        color="#1565C0", linewidth=2, marker="o", markersize=3)
ax.fill_between(monthly["Month"], monthly["Average Daily Boardings"],
                alpha=0.12, color="#1565C0")
ax.set_title("SFMTA — System-Wide Total Average Daily Boardings Over Time",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Total Avg Daily Boardings (all routes)")
ax.grid(axis="y", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Visualization: Top 20 Routes by Average Daily Boardings (most recent month)

latest_month = rd["Month"].max()
latest = rd[rd["Month"] == latest_month]
top_routes = latest.groupby("Route")["Average Daily Boardings"].sum().sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
colors_bar = sns.color_palette("Blues_r", len(top_routes))
ax.barh(top_routes.index[::-1], top_routes.values[::-1],
        color=colors_bar[::-1], edgecolor="white")
ax.set_title(f"Top 20 Routes by Avg Daily Boardings ({latest_month.strftime('%b %Y')})",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Average Daily Boardings")
ax.grid(axis="x", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Visualization: Ridership by Service Category

cat_avg = rd.groupby("Service Category")["Average Daily Boardings"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 4))
colors_cat = sns.color_palette("muted", len(cat_avg))
ax.bar(cat_avg.index, cat_avg.values, color=colors_cat, edgecolor="white")
ax.set_title("Mean Avg Daily Boardings by Service Category", fontsize=13, fontweight="bold")
ax.set_ylabel("Mean Avg Daily Boardings")
ax.tick_params(axis="x", rotation=25)
ax.grid(axis="y", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

## File 5: San Francisco Land Use 2023
**File:** `San_Francisco_Land_Use_-_2023_20260327.csv`  
**Description:** Parcel-level land use data for San Francisco, published by the SF Planning Department for 2023. Each row is a mapped block-lot (parcel) with attributes including residential type, square footage by use category (residential, CIE, medical, MIPS, retail, PDR, visitor), residential unit count, general land use classification, and street address. The geometry column contains MULTIPOLYGON coordinates for each parcel. This file is the spatial backbone for neighbourhood-level density, mixed-use, and transit-accessibility analysis.

In [ ]:
# DF Setup
lu_df = dfs["land_use"]

In [ ]:
# Shape and Columns of Land Use Data
print(f"Total Parcels  : {len(lu_df):,}")
print(f"Columns        : {list(lu_df.columns)}")

In [ ]:
# Notable Stats
print(f"Unique Land Use Types : {lu_df['landuse'].nunique()}")
print(f"Unique Res Types      : {lu_df['restype'].nunique()}")
print(f"Total Res Units       : {pd.to_numeric(lu_df['resunits'].astype(str).str.replace(',',''), errors='coerce').sum():,.0f}")
print()
print("Land Use Breakdown:")
print(lu_df["landuse"].value_counts().to_string())

In [ ]:
# Display Land Use (head)
display(lu_df.drop(columns=["the_geom"], errors="ignore").head(5))

In [ ]:
# Visualization: Donut Chart — Land Use Type Distribution

lu_counts = lu_df["landuse"].value_counts()

colors_lu = sns.color_palette("tab10", len(lu_counts))
fig, ax = plt.subplots(figsize=(8, 6))
wedges, texts, autotexts = ax.pie(
    lu_counts,
    labels=lu_counts.index,
    autopct="%1.1f%%",
    startangle=140,
    colors=colors_lu,
    wedgeprops=dict(width=0.5, edgecolor="white", linewidth=2),
    textprops={'fontsize': 9}
)
for at in autotexts:
    at.set_fontsize(8)
    at.set_fontweight("bold")
ax.set_title("San Francisco — Parcel Land Use Type Distribution (2023)",
             fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Visualization: Top Residential Types by Parcel Count

res_counts = lu_df[lu_df["landuse"] == "RESIDENT"]["restype"].value_counts().head(15)

fig, ax = plt.subplots(figsize=(10, 5))
colors_res = sns.color_palette("Greens_r", len(res_counts))
ax.bar(res_counts.index, res_counts.values, color=colors_res, edgecolor="white")
ax.set_title("Top Residential Sub-Types by Parcel Count (2023)", fontsize=13, fontweight="bold")
ax.set_ylabel("Number of Parcels")
ax.tick_params(axis="x", rotation=35)
ax.grid(axis="y", alpha=0.4)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Visualization: Parcel Scatter Map (lat/lon extracted from centroid)

import re

def extract_centroid(geom_str):
    """Approximate centroid from first polygon ring of a MULTIPOLYGON."""
    try:
        coords = re.findall(r"(-?\d+\.\d+)\s+(-?\d+\.\d+)", str(geom_str))
        if coords:
            lons = [float(c[0]) for c in coords]
            lats = [float(c[1]) for c in coords]
            return sum(lons)/len(lons), sum(lats)/len(lats)
    except:
        pass
    return np.nan, np.nan

# Sample for speed
sample_lu = lu_df.sample(min(10_000, len(lu_df)), random_state=42).copy()
sample_lu[["lon", "lat"]] = sample_lu["the_geom"].apply(
    lambda g: pd.Series(extract_centroid(g))
)
sample_lu = sample_lu.dropna(subset=["lon", "lat"])

# Colour by land use
lu_types = sample_lu["landuse"].unique()
palette = dict(zip(lu_types, sns.color_palette("tab10", len(lu_types))))

fig, ax = plt.subplots(figsize=(8, 9))
for lu_type, grp in sample_lu.groupby("landuse"):
    ax.scatter(grp["lon"], grp["lat"], s=1.5, alpha=0.35,
               color=palette[lu_type], label=lu_type, linewidths=0)
ax.set_title("SF Parcels — Land Use Map (sampled, 2023)", fontsize=14, fontweight="bold")
ax.set_xlabel("Longitude", fontsize=10)
ax.set_ylabel("Latitude", fontsize=10)
ax.set_facecolor("#F5F5F5")
ax.grid(True, linestyle="--", alpha=0.4)
ax.legend(markerscale=5, fontsize=8, loc="lower left")
plt.tight_layout()
plt.show()

## Summary Statistics
A consolidated overview of all modifier datasets.

In [ ]:
print("  SF Modifier Data — Dataset Summary")
print()

# Commute time
ct_total = commute_df.iloc[0, 1] if len(commute_df) > 0 else 'N/A'
print(f"  ACS Commute Departure Times")
print(f"    Departure Bands      : {len(dfs['commute_time']) - 1}")
print(f"    Total Workers (est.) : {ct_total}")
print()

# LODES
print(f"  LODES Origin-Destination (JT05 2023)")
print(f"    Total Records        : {len(od_df):,}")
print(f"    Unique Work Blocks   : {od_df['w_geocode'].nunique():,}")
print(f"    Unique Home Blocks   : {od_df['h_geocode'].nunique():,}")
print(f"    Total Jobs (S000)    : {od_df['S000'].sum():,}")
print()

# Population
pop_total = dfs['population'].iloc[0, 1] if len(dfs['population']) > 0 else 'N/A'
print(f"  Decennial Census 2020 (P1)")
print(f"    Total Population     : {pop_total}")
print(f"    Race Categories      : {len(dfs['population']) - 1}")
print()

# Ridership
print(f"  SFMTA Ridership by Route")
print(f"    Total Records        : {len(rd):,}")
print(f"    Unique Routes        : {rd['Route'].nunique()}")
print(f"    Date Range           : {rd['Month'].min().strftime('%b %Y')} → {rd['Month'].max().strftime('%b %Y')}")
print()

# Land Use
print(f"  SF Land Use 2023")
print(f"    Total Parcels        : {len(lu_df):,}")
print(f"    Land Use Types       : {lu_df['landuse'].nunique()}")
print()

# Land use breakdown
print("Land Use Breakdown:")
for lu_type, count in lu_df["landuse"].value_counts().items():
    print(f"  {lu_type:<20}: {count:,}")

## References
List out the resources and references used for cultivating this exploration file.

- **ACS Table C08132:** https://data.census.gov/table/ACSDT1Y2024.C08132
- **LODES LEHD Origin-Destination:** https://lehd.ces.census.gov/data/
- **Decennial Census PL 94-171:** https://www.census.gov/programs-surveys/decennial-census/about/rdo/summary-files.html
- **SFMTA Ridership Data:** https://www.sfmta.com/reports/ridership-statistics
- **SF Land Use Dataset:** https://data.sfgov.org/Housing-and-Buildings/Land-Use/us3s-fp9q
